<a href="https://colab.research.google.com/github/LoneWolf206/sentiment-analyzer/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json again

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d snap/amazon-fine-food-reviews
!unzip amazon-fine-food-reviews.zip

In [ ]:
!unzip amazon-fine-food-reviews.zip

In [ ]:
import pandas as pd
df = pd.read_csv('Reviews.csv')
print(df.shape)
print(df.head())

In [ ]:
print(df['Score'].value_counts())
print(df['Text'].isnull().sum())


In [ ]:
print(df.describe())
print(df.info())

In [ ]:
# Convert scores to sentiment labels
def get_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(get_sentiment)
print(df['sentiment'].value_counts())

In [ ]:

# Sample equal numbers from each class
min_count = df['sentiment'].value_counts().min()
df_balanced = df.groupby('sentiment').sample(n=min_count, random_state=42)

print(df_balanced['sentiment'].value_counts())
print(df_balanced.shape)

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove special characters
    text = text.lower().strip()               # lowercase
    return text

df_balanced['clean_text'] = df_balanced['Text'].apply(clean_text)
print(df_balanced['clean_text'].head())

In [ ]:
df_sample = df_balanced.sample(n=15000, random_state=42)
print(df_sample['sentiment'].value_counts())

In [ ]:
!pip install transformers torch
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Test on a sample
sample = df_sample['clean_text'].iloc[0]
tokens = tokenizer(sample, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
print(tokens)

In [ ]:
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df_sample['label'] = df_sample['sentiment'].map(label_map)

In [ ]:
from torch.utils.data import Dataset
import torch

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

texts = df_sample['clean_text'].tolist()
labels = df_sample['label'].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

train_dataset = SentimentDataset(X_train, y_train, tokenizer)
val_dataset = SentimentDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from torch.nn import CrossEntropyLoss

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss/len(loader), correct/total

# Train for 3 epochs
for epoch in range(3):
    loss, acc = train_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.4f}")

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct/total

val_acc = evaluate(model, val_loader, device)
print(f"Validation Accuracy: {val_acc:.4f}")

In [ ]:
model.save_pretrained('sentiment_bert')
tokenizer.save_pretrained('sentiment_bert')
print("Model saved")

In [ ]:
def predict(text, model, tokenizer, device):
    model.eval()
    encoding = tokenizer(text, max_length=128, truncation=True,
                        padding='max_length', return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    pred = outputs.logits.argmax(dim=1).item()
    labels = {0: 'negative', 1: 'neutral', 2: 'positive'}
    return labels[pred]

# Test it
print(predict("This product is absolutely amazing!", model, tokenizer, device))
print(predict("Terrible quality, waste of money", model, tokenizer, device))
print(predict("It's okay, nothing special", model, tokenizer, device))